# CMIP6 CNRM-CM6-1-HR sea-ice concentration: xarray inspection

When: 2026-10-22

Work directly with the historical monthly `siconc` source file using xarray. This complements the [subset memory-error notebook](cds-error-2026-10-21-cmip6-cnrm-cm6-1-hr-siconc-subset-memory.ipynb).

Run this notebook in an environment containing xarray and netCDF4. `base_dir` points to the local copy; the original archive directory is retained as a commented alternative. The inspection cells show the source metadata; the memory comparison below has not been executed here.


In [1]:
from pathlib import Path

import xarray as xr

base_dir = Path("~/tmp/c3s-cmip6").expanduser()
# base_dir = Path(
#     "/work/ik1017/CMIP6/data/CMIP6/CMIP/CNRM-CERFACS/"
#     "CNRM-CM6-1-HR/historical/r1i1p1f2/SImon/siconc/gn/v20191021"
# )
source = base_dir / "siconc_SImon_CNRM-CM6-1-HR_historical_r1i1p1f2_gn_185001-201412.nc"
if not source.is_file():
    raise FileNotFoundError(f"Source file not found; check base_dir: {source}")


## Open and inspect

`chunks=None` uses xarray's lazy backend indexing without requiring Dask. Opening reads metadata and dimension coordinates; it does not load the full `siconc` array. If Dask is installed, change `CHUNKS` to `{"time": 1}` to experiment with monthly time chunks. Storage chunks can still affect actual read sizes.

Avoid calling `.values`, `.load()`, or `.compute()` on the full dataset during this inspection.


In [2]:
CHUNKS = None

ds = xr.open_dataset(source, engine="netcdf4", chunks=CHUNKS)
ds


<xarray.Dataset> Size: 12GB
Dimensions:      (y: 1050, x: 1442, nvertex: 4, time: 1980, axis_nbounds: 2)
Coordinates:
  * time         (time) datetime64[ns] 16kB 1850-01-16T12:00:00 ... 2014-12-1...
    lat          (y, x) float32 6MB ...
    lon          (y, x) float32 6MB ...
    type         |S255 255B ...
Dimensions without coordinates: y, x, nvertex, axis_nbounds
Data variables:
    bounds_lon   (y, x, nvertex) float32 24MB ...
    bounds_lat   (y, x, nvertex) float32 24MB ...
    time_bounds  (time, axis_nbounds) datetime64[ns] 32kB ...
    siconc       (time, y, x) float32 12GB ...
Attributes: (12/52)
    Conventions:            CF-1.7 CMIP-6.2
    creation_date:          2019-05-23T13:08:52Z
    description:            CMIP6 historical
    title:                  CNRM-CM6-1-HR model output prepared for CMIP6 and...
    activity_id:            CMIP
    contact:                contact.cmip@meteo.fr
    ...                     ...
    dr2xml_md5sum:          45d4369d889ddfb8149d771d8625e9ec
    xios_commit:            1442-shuffle
    nemo_gelato_commit:     84a9e3f161dade7_8250e198106a168
    arpege_minor_version:   6.3.3
    history:                none
    tracking_id:            hdl:21.14100/379dc5e4-51cd-433c-ba51-01d48511e997

In [3]:
print("Dimensions:", dict(ds.sizes))
print("Time coverage:", ds.time.values[0], "to", ds.time.values[-1])
print("Time index is increasing:", ds.indexes["time"].is_monotonic_increasing)
print("Time index is unique:", ds.indexes["time"].is_unique)

for name, variable in ds.variables.items():
    print(
        name,
        "dims=", variable.dims,
        "shape=", variable.shape,
        "dtype=", variable.dtype,
        "storage_chunks=", variable.encoding.get("chunksizes"),
        "dask_chunks=", variable.chunks,
    )


Dimensions: {'y': 1050, 'x': 1442, 'nvertex': 4, 'time': 1980, 'axis_nbounds': 2}
Time coverage: 1850-01-16T12:00:00.000000000 to 2014-12-16T12:00:00.000000000
Time index is increasing: True
Time index is unique: True
bounds_lon dims= ('y', 'x', 'nvertex') shape= (1050, 1442, 4) dtype= float32 storage_chunks= (1050, 1442, 4) dask_chunks= None
bounds_lat dims= ('y', 'x', 'nvertex') shape= (1050, 1442, 4) dtype= float32 storage_chunks= (1050, 1442, 4) dask_chunks= None
time_bounds dims= ('time', 'axis_nbounds') shape= (1980, 2) dtype= datetime64[ns] storage_chunks= (1, 2) dask_chunks= None
siconc dims= ('time', 'y', 'x') shape= (1980, 1050, 1442) dtype= float32 storage_chunks= (1, 1050, 1442) dask_chunks= None
lat dims= ('y', 'x') shape= (1050, 1442) dtype= float32 storage_chunks= (1050, 1442) dask_chunks= None
lon dims= ('y', 'x') shape= (1050, 1442) dtype= float32 storage_chunks= (1050, 1442) dask_chunks= None
type dims= () shape= () dtype= |S255 storage_chunks= None dask_chunks= None


## Select time before loading data

Start with 1979–1980, then load just one monthly field. Adjust the dates as needed. The printed sizes describe decoded array sizes, not peak process memory. Inspect the actual latitude and longitude dimensions above before adding any spatial selection; no grid structure is assumed here.


In [4]:
siconc = ds["siconc"]
subset = siconc.sel(time=slice("1979-01-01", "1980-12-31"))

print(f"Full variable: {siconc.nbytes / 2**30:.2f} GiB decoded")
print(f"Selected interval: {subset.nbytes / 2**20:.2f} MiB decoded")
subset


Full variable: 11.17 GiB decoded
Selected interval: 138.62 MiB decoded


<xarray.DataArray 'siconc' (time: 24, y: 1050, x: 1442)> Size: 145MB
[36338400 values with dtype=float32]
Coordinates:
  * time     (time) datetime64[ns] 192B 1979-01-16T12:00:00 ... 1980-12-16T12...
    lat      (y, x) float32 6MB ...
    lon      (y, x) float32 6MB ...
    type     |S255 255B ...
Dimensions without coordinates: y, x
Attributes:
    online_operation:    average
    cell_methods:        area: mean where sea time: mean
    interval_operation:  900 s
    interval_write:      1 month
    standard_name:       sea_ice_area_fraction
    description:         Sea Ice Area Fraction (Ocean Grid)
    long_name:           Sea Ice Area Fraction (Ocean Grid)
    history:             none
    units:               %
    cell_measures:       area: areacello

In [5]:
# Load only the first selected monthly field for interactive work.
sample = subset.isel(time=0).load()
sample


<xarray.DataArray 'siconc' (y: 1050, x: 1442)> Size: 6MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(1050, 1442), dtype=float32)
Coordinates:
    lat      (y, x) float32 6MB -78.54 -78.54 -78.54 -78.54 ... 50.02 50.0 50.0
    lon      (y, x) float32 6MB 72.75 73.0 73.25 73.5 ... 73.01 73.01 73.0 73.0
    type     |S255 255B b'typesi'
    time     datetime64[ns] 8B 1979-01-16T12:00:00
Dimensions without coordinates: y, x
Attributes:
    online_operation:    average
    cell_methods:        area: mean where sea time: mean
    interval_operation:  900 s
    interval_write:      1 month
    standard_name:       sea_ice_area_fraction
    description:         Sea Ice Area Fraction (Ocean Grid)
    long_name:           Sea Ice Area Fraction (Ocean Grid)
    history:             none
    units:               %
    cell_measures:       area: areacello

## Use case: area masking before time selection

Compare all three cases for 1979–1980 and `area="-180,39,180,90"`:

1. Select time only.
2. Select time, then apply the area mask.
3. Apply the area mask, then select time.

The file has two-dimensional `lat(y, x)` and `lon(y, x)` coordinates. The helper below uses xarray's `.where()` to mask `siconc` and drop spatial rows and columns outside the area. With `chunks=None`, masking before selecting time can load data across all 1,980 source timesteps instead of the requested 24.

Each case opens the file again and loads its result. All three run unconditionally. The area-first case can use several GB of memory. This is a small xarray demonstration of the suspected allocation mechanism, rather than the complete clisops or Rook workflow.

Memory measurements use `memory_profiler`; install it in the notebook environment with `%pip install memory_profiler` if needed.


In [6]:
from memory_profiler import memory_usage

START = "1979-01-01"
END = "1980-12-31"
AREA = (-180.0, 39.0, 180.0, 90.0)  # west, south, east, north


def subset_area(data):
    west, south, east, north = AREA
    lon = (data.lon + 180) % 360 - 180
    mask = (
        (data.lat >= south) & (data.lat <= north)
        & (lon >= west) & (lon <= east)
    )
    return data.where(mask, drop=True)


### 1. Time only


In [7]:
def time_only():
    with xr.open_dataset(source, engine="netcdf4", chunks=None) as dataset:
        result = dataset.siconc.sel(time=slice(START, END))
        result.load()
        return result.nbytes / 2**20


### 2. Time before area


In [8]:
def time_then_area():
    with xr.open_dataset(source, engine="netcdf4", chunks=None) as dataset:
        selected = dataset.siconc.sel(time=slice(START, END))
        result = subset_area(selected)
        result.load()
        return result.nbytes / 2**20


### 3. Area before time


In [9]:
def area_then_time():
    with xr.open_dataset(source, engine="netcdf4", chunks=None) as dataset:
        selected = subset_area(dataset.siconc)
        result = selected.sel(time=slice(START, END))
        result.load()
        return result.nbytes / 2**20


### Run all three and compare memory

The table shows the kernel's memory before each case, its sampled peak during the case, and the decoded result size. Sampling can miss very short peaks, and Python may retain memory between cases; these are indicative measurements in the same kernel, not isolated process peaks. `max_iterations=1` makes the profiler execute each case once.


In [10]:
print(f"{'Case':<20} {'Before (MiB)':>14} {'Peak (MiB)':>12} {'Result (MiB)':>14}")

for case in [time_only, time_then_area, area_then_time]:
    before = memory_usage()[0]
    peak, result_size = memory_usage(
        (case, (), {}),
        interval=0.1,
        max_usage=True,
        retval=True,
        max_iterations=1,
    )
    print(f"{case.__name__:<20} {before:>14.1f} {peak:>12.1f} {result_size:>14.1f}")


Case                   Before (MiB)   Peak (MiB)   Result (MiB)
time_only                     245.7        657.5          138.6
time_then_area                407.7        550.6           46.6
area_then_time                542.6       7202.5           46.6


The two area cases should return the same decoded result size. A much higher peak for `area_then_time` illustrates how masking the source before selecting time can consume much more memory than the final result. It supports the suspected mechanism but does not establish the cause of the service failure.


## Close the file

Run this cell when finished with lazy operations on `ds` and `subset`. The loaded `sample` remains available in memory.


In [11]:
ds.close()
